# Gemma 4 12B QLoRA fine-tuning for CEFR classification (text -> label)

Same train/val splits as the encoder runs. Test set stays held out.
Predictions are scored by constrained label likelihood, so `p_pred` is
directly comparable with the encoder probability panel.

In [2]:
import os, json, csv
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

from metrics import evaluate_predictions, LABEL_NAMES

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)


@dataclass
class CFG:
    model_id: str = "unsloth/gemma-4-12b-it"
    train_path: str = "data/train.jsonl"
    eval_path: str = "data/val.jsonl"
    test_path: str = "data/test.jsonl"      # not touched in this notebook
    text_col: str = "text"
    label_col: str = "label"
    max_seq_length: int = 3072
    load_in_4bit: bool = True

    # LoRA
    lora_r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.0

    # training
    lr: float = 1e-4
    epochs: float = 3.0
    batch_size: int = 2
    grad_accum: int = 8                     # effective batch 16
    warmup_ratio: float = 0.05
    weight_decay: float = 0.01
    seed: int = 42

    run_dir: str = "runs/gemma4_lora"


cfg = CFG()
os.makedirs(cfg.run_dir, exist_ok=True)

## Load data

In [3]:
df_train = pd.read_json(cfg.train_path, lines=True)
df_val = pd.read_json(cfg.eval_path, lines=True)

df_train = df_train[df_train[cfg.label_col].isin(LEVELS)].reset_index(drop=True)
df_val = df_val[df_val[cfg.label_col].isin(LEVELS)].reset_index(drop=True)

print(f"Train: {len(df_train)}  Val: {len(df_val)}")
print(df_train[cfg.label_col].value_counts().sort_index())

Train: 3797  Val: 599
label
A2     142
A2+    573
B1     702
B1+    882
B2     699
B2+    430
C1     296
C1+     73
Name: count, dtype: int64


## Prompts

Same SYSTEM_PROMPT as the prompting notebook, so any difference in results is
attributable to fine-tuning rather than to a different framing of the task.
CLASSIFICATION_REQUEST is simplified: the JSON schema with justification and
feedback fields is dropped, since this run trains on labels only.

In [4]:
SYSTEM_PROMPT = """You are an expert in assessing written English produced by learners of English as a foreign language.
You will assign a CEFR level to learner writing samples using the following official CEFR descriptors for Overall Written Production.

CEFR Overall Written Production Descriptors (Council of Europe, 2020):

C2: Can produce clear, smoothly flowing, complex texts in an appropriate and effective style 
and a logical structure which helps the reader identify significant points.

C1: Can produce clear, well-structured texts of complex subjects, underlining the relevant 
salient issues, expanding and supporting points of view at some length with subsidiary points, 
reasons and relevant examples, and rounding off with an appropriate conclusion.
Can employ the structure and conventions of a variety of genres, varying the tone, style and 
register according to addressee, text type and theme.

B2: Can produce clear, detailed texts on a variety of subjects related to their field of 
interest, synthesising and evaluating information and arguments from a number of sources.

B1: Can produce straightforward connected texts on a range of familiar subjects within their 
field of interest, by linking a series of shorter discrete elements into a linear sequence.

A2: Can produce a series of simple phrases and sentences linked with simple connectors 
like "and", "but" and "because".

A1: Can give information about matters of personal relevance (e.g. likes and dislikes, family, 
pets) using simple words and basic expressions.
Can produce simple isolated phrases and sentences.

Plus levels represent a very strong competence at a level that does not yet reach the minimum 
standard for the next criterion level. Generally, features of the level above are starting to appear.

The levels you should assign are: A2, A2+, B1, B1+, B2, B2+, C1, C1+."""

LEVELS_STR = ", ".join(LEVELS)

CLASSIFICATION_REQUEST = (
    f"Assign a CEFR level to the following learner writing. "
    f"The possible levels are: {LEVELS_STR}.\n\n"
    "Respond with the level only.\n\n"
    "Text to classify:\n\n"
)


def build_messages(text, label=None):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CLASSIFICATION_REQUEST + str(text)},
    ]
    if label is not None:
        msgs.append({"role": "assistant", "content": label})
    return msgs

## Model and LoRA adapters

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.model_id,
    max_seq_length=cfg.max_seq_length,
    dtype=None,
    load_in_4bit=cfg.load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg.seed,
)

c:\Users\coope\AppData\Local\Programs\Python\Python312\Lib\site-packages\unsloth\__init__.py:1554: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0911 21:41:33.712000 20248 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0911 21:41:33.759000 20248 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


c:\Users\coope\AppData\Local\Programs\Python\Python312\Lib\site-packages\unsloth\import_fixes.py:2124: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


==((====))==  Unsloth 2026.9.4: Fast Gemma4_Unified patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.842 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Gemma 4 has no separate system role in its chat template, so
`apply_chat_template` folds the system message into the first user turn.
Check the rendered output below before training.

In [6]:
# %%
model.print_trainable_parameters()

trainable params: 131,137,536 || all params: 12,090,867,712 || trainable%: 1.0846


In [7]:
# %%
print(type(tokenizer).__name__)
tok = getattr(tokenizer, "tokenizer", tokenizer)
s = tok.apply_chat_template(build_messages("test"), tokenize=False, add_generation_prompt=True)
print(repr(s[-200:]))

Gemma4UnifiedProcessor
'l to the following learner writing. The possible levels are: A2, A2+, B1, B1+, B2, B2+, C1, C1+.\n\nRespond with the level only.\n\nText to classify:\n\ntest<turn|>\n<|turn>model\n<|channel>thought\n<channel|>'


In [9]:
# %%
from datasets import Dataset

EOT = "<turn|>"

def to_training_text(row):
    prompt = tok.apply_chat_template(
        build_messages(row[cfg.text_col]),
        tokenize=False,
        add_generation_prompt=True,
    )
    return prompt + row[cfg.label_col] + EOT


print(repr(to_training_text(df_train.iloc[0])[-80:]))

train_ds = Dataset.from_list(
    [{"text": to_training_text(r)} for _, r in df_train.iterrows()])
eval_ds = Dataset.from_list(
    [{"text": to_training_text(r)} for _, r in df_val.iterrows()])

' you with my parents.<turn|>\n<|turn>model\n<|channel>thought\n<channel|>A2+<turn|>'


## Trainer

`train_on_responses_only` masks the loss to the label tokens. Without it the
prompt tokens dominate the loss and almost nothing is learned about the task.

In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=cfg.max_seq_length,
        per_device_train_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        num_train_epochs=cfg.epochs,
        learning_rate=cfg.lr,
        warmup_ratio=cfg.warmup_ratio,
        weight_decay=cfg.weight_decay,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        output_dir=cfg.run_dir,
        seed=cfg.seed,
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/3797 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/3797 [00:00<?, ? examples/s]

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

### Confirm the loss mask

Should print the CEFR label only and end of turn tag. If the whole prompt appears, the marker
strings above do not match the Gemma 4 template and need fixing.

In [11]:
_s = trainer.train_dataset[0]
_kept = [t for t in _s["labels"] if t != -100]
print(repr(tokenizer.tokenizer.decode(_kept)))

'<|channel>thought\n<channel|>A2+<turn|>'


In [12]:
stats = trainer.train()
print(stats.metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,797 | Num Epochs = 3 | Total steps = 714
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 131,137,536 of 12,090,867,712 (1.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.159228,10.712162
2,0.191391,0.192659
3,0.166168,0.159691


Filter:   0%|          | 0/599 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in runs/gemma4_lora\checkpoint-238\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in runs/gemma4_lora\checkpoint-476\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in runs/gemma4_lora\checkpoint-714\tokenizer_config.json.


{'train_runtime': 4882.512, 'train_samples_per_second': 2.333, 'train_steps_per_second': 0.146, 'total_flos': 5.310729541858099e+17, 'train_loss': 0.3393176240580423, 'epoch': 3.0}


#### Save the model

In [14]:
# %%
model.save_pretrained("runs/gemma4_lora/lora_adapter")
tok.save_pretrained("runs/gemma4_lora/lora_adapter")

('runs/gemma4_lora/lora_adapter\\tokenizer_config.json',
 'runs/gemma4_lora/lora_adapter\\chat_template.jinja',
 'runs/gemma4_lora/lora_adapter\\tokenizer.json')